In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

import matplotlib.pyplot as plt

pi = 3.14159265359
maxval=1e9
minval=1e-9

2026-04-15 21:52:18.983809: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from DG.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import custom_loss
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler
# from models.models import CreateModel # Conv2D model

In [5]:
def create_tfrecords(NOISE_MU=0.0, NOISE_SIGMA=0.0):

    dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/"
    tfrecords_base_dir = "/uscms/home/jennetd/nobackup/smart-pixels/tfrecords"

    dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'train_contained/')
    dataset_dir_val   = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

    tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3sr_16x16_'+str(int(NOISE_SIGMA))+'eNoise_train')
    tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'3sr_16x16_'+str(int(NOISE_SIGMA))+'eNoise_test')

    batch_size = 5000
    val_batch_size = 5000
    train_file_size = len(os.listdir(dataset_dir_train))
    val_file_size = len(os.listdir(dataset_dir_val))

    start_time = time.time()
    validation_generator = OptimizedDataGenerator(
        dataset_base_dir = dataset_dir_val,
        file_type = "parquet",
        data_format = "3D",
        batch_size = val_batch_size,
        # optimize_batch_size = True,
        file_count = val_file_size,
        to_standardize= False,
        select_contained = True,
        noise = [NOISE_MU,NOISE_SIGMA], #[mean, sigma]
        min_threshold = None,
        max_threshold = None,
        labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
        input_shape = (2,16,16), # (20,13,21),
        transpose = (0,2,3,1),
        shuffle = False, 
        files_from_end=True,
        tfrecords_dir = tfrecords_dir_val,
        use_time_stamps = [0,19],
        max_workers = 2
    )

    print("--- Validation generator %s seconds ---" % (time.time() - start_time))

    # training generator
    start_time = time.time()
    training_generator = OptimizedDataGenerator(
        dataset_base_dir = dataset_dir_train,
        file_type = "parquet",
        data_format = "3D",
        batch_size = batch_size,
        # optimize_batch_size = True,
        file_count = train_file_size,
        to_standardize= False,
        select_contained = True,
        noise = [NOISE_MU,NOISE_SIGMA], #[mean, sigma]
        min_threshold = None,
        max_threshold = None,
        labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
        input_shape = (2,16,16), # (20,13,21),
        transpose = (0,2,3,1),
        shuffle = False, # True 
        tfrecords_dir = tfrecords_dir_train,
        use_time_stamps = [0,19],
        max_workers = 2
    )
    print("--- Training generator %s seconds ---" % (time.time() - start_time))
    

In [6]:
create_tfrecords(0.0, 40.0)

Processing Files...: 100%|██████████| 20/20 [00:06<00:00,  2.96it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_40eNoise_test does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 21/21 [00:09<00:00,  2.28it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_40eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_40eNoise_test/metadata.json
--- Validation generator 16.27048349380493 seconds ---


Processing Files...: 100%|██████████| 80/80 [00:27<00:00,  2.86it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_40eNoise_train does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 84/84 [00:35<00:00,  2.36it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_40eNoise_train/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_40eNoise_train/metadata.json


--- Training generator 63.957552909851074 seconds ---


In [7]:
create_tfrecords(0.0, 120.0)

Processing Files...: 100%|██████████| 20/20 [00:05<00:00,  3.58it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_120eNoise_test does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 21/21 [00:08<00:00,  2.49it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_120eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_120eNoise_test/metadata.json
--- Validation generator 14.431134462356567 seconds ---


Processing Files...: 100%|██████████| 80/80 [00:21<00:00,  3.67it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_120eNoise_train does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 84/84 [00:36<00:00,  2.32it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_120eNoise_train/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_120eNoise_train/metadata.json
--- Training generator 58.35236573219299 seconds ---
